# 02 — Reading the disturbed anticentre

**Scientific question:** Before invoking Sagittarius or a tidal model, what do Lambert's released data actually tell us about Monoceros and the Anticenter Stream?

> **Data boundary.** This notebook uses downstream figure products released by Lambert for Zenodo record **18236902** (DOI `10.5281/zenodo.18236902`). These products are **not** the unpublished DESI DR2 source catalogue. Most maps are fixed author-binned arrays; only the Figure 13/14 file supplies individual rows, and those rows already passed the authors' upstream selection.

In [ ]:
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from matplotlib.colors import LogNorm
from scipy.stats import spearmanr

from lambert_lab.data import load_figure13_stars, load_lambert_inventory, load_released_image
from lambert_lab.plotting import constant_lz_curves, display_released_image, zero_centered_norm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
VALIDATION_DIR = ROOT / 'figures' / 'validation'
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
inventory = load_lambert_inventory(ROOT)
release = inventory['release']
print(f"Release: {release['title']}")
print(f"DOI: {release['doi']}; cached archive SHA-256: {release['archive']['sha256']}")

## First: what kind of data can we operate on?

A pixel map and a star table permit different questions. A fixed 2-D map lets us display released values, preserve their orientation, and compare morphology. It does **not** let us change the source selection, recover exact bin edges, rebin, recompute a median, or alter an embedded minimum-count mask. A row table lets us filter released rows by a column that is actually present—but it still cannot restore columns or stars removed upstream.

> 🔭 **Observable / released measurement:** author-produced 2-D aggregates for Figures 5, 6, and 8; selected individual rows for Figures 13/14.  
> 🔧 **Computational operation here:** checksum validation, direct display with `origin='lower'`, and later an explicit sign cut on released per-star $V_R$.

In [ ]:
map_names = [
    'lb_VR_fig5.fits', 'lb_VZ_fig5.fits',
    'XY_overdensity_fig6.fits', 'XY_VR_fig6.fits', 'XY_VZ_fig6.fits',
    'RVphi_count_fig8.fits',
]
maps = {name: load_released_image(ROOT, name) for name in map_names}
summary = Table(
    rows=[(p.figure, p.panel, p.filename, str(p.data.shape), p.value_kind, p.value_unit)
          for p in maps.values()],
    names=['Figure', 'panel', 'released file / HDU 0', 'shape', 'representation', 'value unit'],
)
summary

# Act 1 — The same outer disc, different kinematics

### Question
How can two structures overlap on the sky yet occupy different velocity regimes?

### Physical intuition
A density map asks *where are there more stars?* A median-velocity map asks *how does the typical star in each retained bin move?* The second question can expose coherent populations even where projected densities overlap. Lambert uses $V_R>0$ for motion outward from the Galactic centre and $V_Z>0$ for motion toward the North Galactic Pole.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True, sharey=True)
for ax, filename, symbol in zip(
    axes, ['lb_VR_fig5.fits', 'lb_VZ_fig5.fits'], [r'$V_R$', r'$V_Z$']
):
    product = maps[filename]
    image = display_released_image(
        ax, product, cmap='coolwarm', norm=zero_centered_norm(product.data),
        title=f"Figure 5 released median {symbol}",
    )
    fig.colorbar(image, ax=ax, label=f'{symbol} [km s$^{{-1}}$]')
fig.suptitle(r'Sky view: $R>14$ kpc MSTO anticentre sample', y=1.02)
fig.tight_layout()
fig.savefig(VALIDATION_DIR / 'notebook02_figure5_sky_kinematics.png', dpi=140, bbox_inches='tight')
plt.show()

### How to read these maps

- **Axes:** Galactic longitude $l$ and latitude $b$, both in degrees. The displayed range is $150^\circ<l<220^\circ$, $20^\circ<b<40^\circ$.
- **Population/product:** Lambert's MSTO anticentre selection with $R>14$ kpc. Every pixel is an **already-computed author median** in a retained bin. NaN pixels are transparent; the release does not say whether every NaN is empty or otherwise masked.
- **What to notice:** across much of the map, the highest-latitude ACS-associated band (most clearly around $b\sim35$–$38^\circ$) is redder—positive $V_R$ and $V_Z$—while the lower-latitude Monoceros-associated region is predominantly bluer—negative signs. Individual pixels vary; the coherent regimes, not a hard latitude boundary, are the point.
- **Direct statement:** within this released, selected region, the named structures are associated with opposite-sign radial and vertical kinematics.
- **What we cannot say:** we did not recompute these medians. Source rows, exact image bin edges, per-bin counts, uncertainties, and source-level cuts are absent. The 42-by-12 shape also conflicts with the caption's stated one-bin-per-degree angular binning, so the inventory uses the released shape and header extents without inventing edges.

> 📈 **Result:** spatial overlap does not imply kinematic identity.  
> 🧠 **Data-supported inference:** ACS-associated and Monoceros-associated stars do not behave as one uniform velocity population in this representation.  
> ⚠️ **Not yet:** no perturber or tidal mechanism follows from a velocity-sign contrast alone.

### Change to a top-down view

Figure 6 selects $0<Z<5$ kpc and displays Galactocentric Cartesian $X$–$Y$. Its left panel is not a raw count: it is Lambert's completeness-corrected density **minus an author-fitted exponential-disc model**. Positive values mean an excess relative to that model. The subtraction is already embedded in the release and is itself model-dependent.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True, sharey=True)
figure6 = [
    ('XY_overdensity_fig6.fits', 'density residual', 'effective count'),
    ('XY_VR_fig6.fits', r'median $V_R$', r'km s$^{-1}$'),
    ('XY_VZ_fig6.fits', r'median $V_Z$', r'km s$^{-1}$'),
]
for ax, (filename, title, unit) in zip(axes, figure6):
    product = maps[filename]
    image = display_released_image(
        ax, product, cmap='coolwarm', norm=zero_centered_norm(product.data), title=title
    )
    fig.colorbar(image, ax=ax, label=unit)
fig.suptitle(r'Figure 6 fixed author products: $0<Z<5$ kpc', y=1.02)
fig.tight_layout()
fig.savefig(VALIDATION_DIR / 'notebook02_figure6_xy_products.png', dpi=140, bbox_inches='tight')
plt.show()

### What changed—and what did not

The axes are now Galactocentric $X$ and $Y$ in kpc, so we see the disc from above rather than on the sky. The density residual locates broad over- and under-densities relative to the authors' smooth model. The velocity maps show coherent red/blue patches that are not identical to the density contrast. This is the workflow lesson: **density finds concentrations; kinematics tests whether the concentrations exhaust the moving population.**

The arrays have shape 16 by 28 over header-stated extents $8<X<23$ kpc and $-8<Y<8$ kpc. Figure 6 removed bins with five or fewer stars before release. Counts needed to undo or change that mask are absent, and the stated X bin rate does not exactly match the released shape. We therefore display pixels directly—`origin='lower'`, no transpose, no rebinning, no new mask.

# Act 2 — Change the projection, change what you see

### Question
Why inspect a related MRi-region selection in $R$–$V_\phi$?

For Figure 8, Lambert uses a related MRi-region selection with $0<Z<5$ kpc and $Y>0$ kpc; it should not be assumed to be identical to the selections in the preceding panels. $R$ is cylindrical distance from the Galactic centre and $V_\phi>0$ is disc rotation. Moving to this phase-space plane can align stars that share orbital properties into a ridge. For a star, $L_Z=R\,V_\phi$, so constant positive angular momentum follows $V_\phi=L_Z/R$: a downward-curving guide. The guides below are calculated by us for intuition. They are not released fits and do not assign a mechanism.

In [ ]:
rvphi = maps['RVphi_count_fig8.fits']
finite_count = rvphi.data[np.isfinite(rvphi.data) & (rvphi.data > 0)]
fig, ax = plt.subplots(figsize=(8, 5.3))
image = display_released_image(
    ax, rvphi, cmap='viridis',
    norm=LogNorm(vmin=finite_count.min(), vmax=finite_count.max()),
    title=r'Figure 8 released density in $R$–$V_\phi$',
)
radius = np.linspace(9, 24, 300)
for lz, vphi in constant_lz_curves(radius, [2500, 3000, 3500]).items():
    ax.plot(radius, vphi, '--', lw=1.2, label=fr'our guide: $L_Z={lz:.0f}$ kpc km s$^{{-1}}$')
ax.set_ylim(140, 270)
ax.legend(fontsize=8, loc='lower left')
fig.colorbar(image, ax=ax, label='completeness-corrected number density')
fig.tight_layout()
fig.savefig(VALIDATION_DIR / 'notebook02_figure8_rvphi_ridge.png', dpi=140, bbox_inches='tight')
plt.show()

### What to notice

- **Axes:** $R$ in kpc horizontally; positive, disc-rotation $V_\phi$ in km s$^{-1}$ vertically. The FITS comment confusingly says “negative azimuthal velocity,” but the paper and displayed range establish the positive convention used here; we do not flip the data.
- **Product:** a 32-by-30 author-binned, completeness-corrected density for the selected MRi-region sample with $0<Z<5$ kpc and $Y>0$ kpc. Bins below 20 stars were removed upstream.
- **Visual feature:** an elongated ridge is easier to recognize here than in sky coordinates. The logarithmic colour normalization changes display contrast only; it does not alter values or bins.
- **Direct statement:** released density is structured in $R$–$V_\phi$. Dashed curves only show how constant $L_Z$ would bend.
- **What cannot be said yet:** resemblance to a guide does not prove a single angular momentum, a tidal spiral, an origin, or a perturbation time. Figure 8's exact $V_\phi$ bin edges are also unavailable (32 rows span a range whose stated rate implies 32.5 bins).

> 🔭 **Observation:** a ridge-like density feature is present in this projection.  
> ⚠️ **Model-dependent interpretation:** connecting that ridge to tidal winding belongs to Notebook 3, not here.

# Act 3 — When does a “stream” stop being an isolated stream?

### Predict before revealing

> **If the ACS were a fully isolated kinematic object, what would you expect to happen to the ACS-like feature outside the narrow visible overdensity?**

A natural prediction is that the characteristic pattern should fade with the narrow high-latitude density enhancement rather than continue smoothly to lower latitude. The Figure 13 table lets us test **kinematic continuity with latitude** because it contains one $l$, $b$, $V_Z$, and $V_R$ value per row. Locating that trend relative to the nominal photometric ACS boundary additionally requires the boundary published in the paper; it is not a column or fit contained in the public table.

In [ ]:
stars = load_figure13_stars(ROOT)

# These are the scientific selections we are independently reapplying.
figure13_longitude = (stars['l'] >= 150*u.deg) & (stars['l'] < 200*u.deg)
vr_positive = stars['V_R'] > 0*u.km/u.s
vr_negative = stars['V_R'] < 0*u.km/u.s

release_counts = (len(stars), int(vr_positive.sum()), int(vr_negative.sum()))
figure13_counts = (
    int(figure13_longitude.sum()),
    int((figure13_longitude & vr_positive).sum()),
    int((figure13_longitude & vr_negative).sum()),
)
print('Entire released table (all, V_R>0, V_R<0):', release_counts)
print('Published Figure 13 longitude range (all, V_R>0, V_R<0):', figure13_counts)
assert release_counts == (7708, 3658, 4050)
assert figure13_counts == (6047, 2769, 3278)

In [ ]:
slice_rows = []
for lo in range(150, 200, 10):
    longitude_slice = (stars['l'] >= lo*u.deg) & (stars['l'] < (lo+10)*u.deg)
    slice_rows.append((
        f'{lo}–{lo+10}', int(longitude_slice.sum()),
        int((longitude_slice & vr_positive).sum()),
        int((longitude_slice & vr_negative).sum()),
    ))
Table(rows=slice_rows, names=['l range [deg]', 'all', 'V_R > 0', 'V_R < 0'])

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 8), sharex=True, sharey=True)
vr_norm = zero_centered_norm(stars['V_R'].to_value(u.km/u.s))
all_row_scatter = None
row_definitions = [
    ('all released rows', np.ones(len(stars), dtype=bool)),
    (r'$V_R>0$ (outward)', vr_positive),
    (r'$V_R<0$ (inward)', vr_negative),
]
for column, lo in enumerate(range(150, 200, 10)):
    longitude_slice = (stars['l'] >= lo*u.deg) & (stars['l'] < (lo+10)*u.deg)
    for row, (label, sign_mask) in enumerate(row_definitions):
        selected = longitude_slice & sign_mask
        ax = axes[row, column]
        scatter_kwargs = dict(s=5, alpha=0.4, linewidths=0, rasterized=True)
        if row == 0:
            scatter_kwargs.update(
                c=stars['V_R'][selected].to_value(u.km/u.s),
                cmap='coolwarm', norm=vr_norm,
            )
        points = ax.scatter(
            stars['b'][selected].to_value(u.deg),
            stars['V_Z'][selected].to_value(u.km/u.s),
            **scatter_kwargs,
        )
        if row == 0:
            all_row_scatter = points
        ax.axhline(0, color='0.3', lw=0.7)
        ax.set_xlim(20, 40)
        ax.set_ylim(-180, 180)
        if row == 0:
            ax.set_title(fr'${lo}^\circ\leq l<{lo+10}^\circ$')
        if column == 0:
            ax.set_ylabel(label + '\n' + r'$V_Z$ [km s$^{-1}$]')
        if row == 2:
            ax.set_xlabel(r'$b$ [deg]')
fig.suptitle('Figure 13-style comparison recomputed from released individual rows', y=0.98)
fig.subplots_adjust(top=0.90, right=0.92, hspace=0.12, wspace=0.12)
colorbar_axis = fig.add_axes([0.94, 0.66, 0.012, 0.21])
fig.colorbar(
    all_row_scatter, cax=colorbar_axis,
    label=r'all-stars row: $V_R$ [km s$^{-1}$]',
)
fig.savefig(VALIDATION_DIR / 'notebook02_figure13_sign_split.png', dpi=140, bbox_inches='tight')
plt.show()

### Reveal: what survives the sign split?

The colour-coded all-stars row previews how radial-velocity sign organizes the $b$–$V_Z$ plane; its colour scale is symmetric and centred exactly on $V_R=0$. The two simple sign rows then make the split explicit. The $V_R>0$ row shows a positive $b$–$V_Z$ trend in all five published 10-degree longitude slices, and that trend continues smoothly to lower $b$ within the released sample. The $V_R<0$ row has a different distribution.

> 🔭 **Released measurement:** 7,708 individual rows from the authors' already-selected MSTO anticentre sample with heliocentric distance $>10$ kpc.  
> 🔧 **Our recomputation:** published $150^\circ\leq l<200^\circ$ slices and strict $V_R>0$ / $V_R<0$ masks. There are no exact-zero $V_R$ rows.  
> 📈 **Directly supported by released rows:** the outward-moving population has a positive $b$–$V_Z$ trend in every plotted longitude slice, extending continuously to lower $b$.  
> 📈 **Supported by combining released rows with the paper's published photometric ACS definition:** ACS-associated kinematics continue below the nominal photometric ACS overdensity.  
> 🧠 **Data-supported inference:** ACS is not behaving like a completely isolated kinematic island in this representation; a named photometric overdensity and a dynamical population need not be identical concepts.

> **Missing published overlays.** The cyan/magenta parabolic ACS boundaries and gray width region in the published Figure 13 cannot be recreated: their coefficients and fit inputs are not in the public release. The photometric-boundary comparison is therefore paper-plus-release evidence, not something derivable from this table alone.

We also cannot reconstruct the original DR2 target/sample selection: the table has no distance, source ID, uncertainties, weights, photometry, metallicity, or selection flags. No boundary is refitted or invented here.

In [ ]:
# Scientific morphology check: the outward subset has a positive b–V_Z rank trend
# in every published longitude slice. This validates a feature, not aesthetics.
trend_rows = []
for lo in range(150, 200, 10):
    selected = (
        (stars['l'] >= lo*u.deg) & (stars['l'] < (lo+10)*u.deg) & vr_positive
    )
    rho = spearmanr(
        stars['b'][selected].to_value(u.deg),
        stars['V_Z'][selected].to_value(u.km/u.s),
    ).statistic
    trend_rows.append((f'{lo}–{lo+10}', int(selected.sum()), rho))
assert all(row[2] > 0 for row in trend_rows)
Table(rows=trend_rows, names=['l range [deg]', 'V_R>0 stars', 'Spearman rho(b, V_Z)'])

The five Spearman coefficients are a **morphology sanity check**: they verify that the visually identified $V_R>0$ trend has the same positive direction in every published longitude slice. They are not a new population-level significance measurement. The public table lacks the upstream selection function and per-star velocity uncertainties, so we do not interpret p-values or make threshold-based significance claims.

## 🧪 Try it yourself—within what the release supports

Choose one of the five published longitude slices and compare its sign rows. Or replace the strict sign mask in the visible cell with a more restrictive threshold such as $V_R>20$ km s$^{-1}$, clearly labeling that as a new experiment on already-selected rows. Do **not** interpret it as a new source-level selection. Rebinning fixed maps, changing their count masks, or reconstructing the ACS photometric boundary is not supported.

## Validation and reproduction ledger

| Paper figure/panel | Released filename / HDU | Representation | What this notebook recomputed | Status | Limitation |
|---|---|---|---|---|---|
| Fig. 5 top left | `lb_VR_fig5.fits` / 0 | 12×42 aggregate | display only | Morphology reproduced: high-$b$ positive versus low-$b$ negative $V_R$ regime | medians, masks, and exact edges cannot be rebuilt; caption/shape binning conflict |
| Fig. 5 top middle | `lb_VZ_fig5.fits` / 0 | 12×42 aggregate | display only | Morphology reproduced: corresponding $V_Z$ sign contrast | same fixed-product limitation |
| Fig. 6 left | `XY_overdensity_fig6.fits` / 0 | 16×28 aggregate | display only | orientation, extent, units, and broad residual morphology reproduced | residual model and ≤5-star mask already embedded |
| Fig. 6 second/right | `XY_VR_fig6.fits`, `XY_VZ_fig6.fits` / 0 | two 16×28 aggregates | display only | coherent signed velocity morphology reproduced | no source rows/counts; X-bin statement conflicts with shape |
| Fig. 8 left | `RVphi_count_fig8.fits` / 0 | 32×30 aggregate | our labeled constant-$L_Z$ guides only | ridge morphology, axes, and positive paper $V_\phi$ convention reproduced | guides are not fits; exact vertical edges and remasking unavailable |
| Fig. 13 | `fig13_and_fig14_table.fits` / 1 | 7,708 selected individual rows | five longitude slices and $V_R$ sign subsets | independently recomputed; counts deterministic; outward $b$–$V_Z$ trend present in every slice | upstream selection and photometric boundary unavailable |

For every fixed map, “reproduced” means correct checksum-backed product, `origin='lower'`, no transpose, header-stated extent, documented units, and matching scientific morphology. It does **not** mean independent reconstruction of author medians or the density model. Exact original image-bin edges are unavailable.

# Evidence ladder: stop where the data stop

### DIRECTLY SUPPORTED / RELEASED OBSERVATION

- ACS- and Monoceros-associated regions show different, opposite-sign radial and vertical kinematics in Lambert's sampled region. This is encoded in fixed author-binned maps; we displayed but did not independently recompute their medians.
- The released Figure 13 selected-star table supports independently reapplying the $V_R$ sign split.

### DIRECTLY SUPPORTED BY RELEASED FIGURE 13 ROWS

- The $V_R>0$ population shows a positive $b$–$V_Z$ trend in all five published longitude slices.
- That trend extends continuously to lower $b$ within the released sample.

### SUPPORTED BY COMBINING THE RELEASED ROWS WITH THE PAPER'S PUBLISHED PHOTOMETRIC ACS DEFINITION

- ACS-associated kinematics continue below the nominal photometric ACS overdensity. The absent boundary coefficients prevent this comparison from being reconstructed from the table alone.

### SUPPORTED INFERENCE

- ACS is not behaving like a completely isolated kinematic island in this representation.
- A named overdensity and a dynamical population need not be identical concepts.

### ⚠️ MODEL-DEPENDENT INTERPRETATION

- ACS as a fold or crest of a broader vertical wave.
- Physical connections to other outer-disc corrugations or the Gaia phase spiral.

### NOT ESTABLISHED HERE

- a unique Sagittarius origin;
- a Monoceros tidal-spiral interpretation;
- perturbation timing.

Those dynamical questions are deferred to Notebook 3 and the discussion. This notebook's durable result is observational: changing projection and conditioning on measured velocity reveals continuity that a narrow density name can hide.

In [ ]:
NOTEBOOK_STAGE = 'task-4-complete'
assert NOTEBOOK_STAGE == 'task-4-complete'
print('Notebook 2 completed offline from checksum-validated Lambert release products.')